# Problém obchodního cestujícího (TSP)

**Zadání:** obchodní cestující má navštívit *n* měst, každé právě jednou, a vrátit se domů. V jakém pořadí města objet,
aby byla cesta co nejkratší?

Zní to jako hlavolam, ale tutéž úlohu denně řeší rozvozová auta, vrtačky desek plošných spojů, skladové roboty
i software pro sekvenování DNA. V tomto notebooku:

1. vytvoříme malou náhodnou mapu měst,
2. najdeme nejlepší trasu **vyzkoušením všech možností** (hrubá síla) a změříme, jak čas exploduje,
3. extrapolujeme: kolik měst zvládneme, než výpočet potrvá déle než stáří vesmíru,
4. ukážeme chytřejší přesnou metodu (**Held–Karp**) – pro velké *n* stále beznadějnou,
5. použijeme **heuristiky**, které rychle dají dobrou (ne nutně nejlepší) trasu.

*Jak číst kód:* toto není předmět o programování. Každá buňka s kódem má komentáře (`# ...`), které lidsky říkají,
co řádky dělají – čtěte především je, přesná syntaxe Pythonu je vedlejší.

In [ ]:
# --- Knihovny (hotové nástroje, které si půjčujeme) ---------------------------
import matplotlib.pyplot as plt                 # kreslení obrázků a grafů
import networkx as nx                           # práce s grafy (města = uzly, silnice = hrany)
import networkx.algorithms.approximation as nx_app   # hotové heuristické algoritmy pro grafy (Christofides)
import math                                     # odmocniny, faktoriály, ...
from time import perf_counter                   # přesné stopky pro měření doby výpočtu
import numpy as np                              # tabulky čísel (matice) a rychlé počítání s nimi
import itertools                                # generování všech permutací / kombinací
import pandas as pd                             # hezky formátované tabulky

## 1. Náhodná mapa měst

Města umístíme na náhodné body v jednotkovém čtverci. Vzdálenost dvou měst je obyčejná vzdálenost vzdušnou čarou
(euklidovská). Všechny vzdálenosti uložíme do **matice vzdáleností**: tabulky, kde řádek *i*, sloupec *j* obsahuje
vzdálenost z města *i* do města *j*. Tahle tabulka je jediné, co algoritmy níže potřebují.

In [ ]:
def tsp_graph(cities: int = 20):
    """Vytvoří `cities` náhodných bodů; vrátí graf, jejich polohy a matici vzdáleností."""
    # random_geometric_graph dá náhodné body v jednotkovém čtverci (seed=3 -> pokaždé stejná mapa)
    G = nx.random_geometric_graph(cities, radius=0.4, seed=3)
    pos = nx.get_node_attributes(G, "pos")      # slovník: číslo města -> souřadnice (x, y)

    # Město 0 je depo ("domov"); dáme ho doprostřed mapy
    pos[0] = (0.5, 0.5)

    # Prázdná tabulka města x města, kterou naplníme vzdálenostmi
    distance_matrix = np.zeros((cities, cities))

    # Pro každou dvojici měst (i, j), kde i < j ...
    for i in range(len(pos)):
        for j in range(i + 1, len(pos)):
            # ... spočítej vzdálenost vzdušnou čarou (Pythagoras: hypot = odmocnina(dx^2 + dy^2))
            dist = math.hypot(pos[i][0] - pos[j][0], pos[i][1] - pos[j][1])
            distance_matrix[i, j] = dist        # ulož do tabulky
            G.add_edge(i, j, weight=dist)       # a přidej do grafu silnici i-j této délky

    # Vyplnili jsme jen horní polovinu tabulky; vzdálenost i->j je stejná jako j->i, tak ji zkopírujeme do dolní
    distance_matrix += distance_matrix.T

    return G, pos, distance_matrix

In [ ]:
def tsp_graph_vis(G, pos, path=[]):
    """Nakreslí mapu. Je-li zadána trasa `path` (seznam měst), nakreslí červeně jen ji; jinak všechny silnice."""
    if not path:
        # žádná trasa -> nakresli všechny silnice tence modře
        nx.draw_networkx_edges(G, pos, edge_color="blue", width=0.5)

    # trasu [0, 3, 1, ..., 0] převeď na seznam sousedních dvojic [(0,3), (3,1), ...]
    edge_list = list(nx.utils.pairwise(path))

    # nakresli města (červená kolečka s čísly) a trasu (tlusté červené čáry)
    nx.draw_networkx(
        G,
        pos,
        with_labels=True,
        edgelist=edge_list,
        node_color='r',
        edge_color="red",
        node_size=200,
        width=3,
    )

In [ ]:
def compute_distance(distance_matrix: np.ndarray, path: list) -> tuple[int, np.ndarray]:
    """Celková délka okružní trasy `path` (seznam čísel měst). Trasa se uzavře automaticky (poslední -> první)."""
    ind_y = path                    # města, kde každý úsek začíná:  [c0, c1, c2, ..., ck]
    ind_x = path[1:] + path[:1]     # města, kde každý úsek končí:   [c1, c2, ..., ck, c0]  (posunuto o jedna, dokola)
    # distance_matrix[ind_y, ind_x] vybere délky všech úseků najednou; .sum() je sečte
    return distance_matrix[ind_y, ind_x].sum(), distance_matrix[ind_y, ind_x]

In [ ]:
# Vytvoř malou instanci se 6 městy, ukaž mapu a matici vzdáleností
start_point = 0
G, pos, distance_matrix = tsp_graph(6)
tsp_graph_vis(G, pos)
pd.DataFrame(distance_matrix)     # pandas zobrazí matici jako čitelnou tabulku

## 2. Hrubá síla: vyzkoušej všechny trasy

Nejjednodušší správný postup: vypiš **všechna** možná pořadí měst, spočítej délku každého, nech si nejkratší.
Když začátek zafixujeme v městě 0, zbylých n−1 měst lze seřadit (n−1)! způsoby (n−1 možností pro první zastávku,
n−2 pro druhou, ...). Pro 6 měst je to 5! = 120 tras – snadné. Brzy uvidíme, co se stane pro 15 nebo 20 měst.

In [ ]:
def brute_force_tsp(distance_matrix: np.ndarray, start: int) -> tuple[list[int], float]:
    """Vyzkouší všechna pořadí měst; vrátí nejkratší trasu a čas výpočtu."""
    points = list(range(distance_matrix.shape[0]))   # všechna čísla měst [0, 1, ..., n-1]
    points.remove(start)                              # start je pevný, permutujeme jen ostatní

    best_distance = np.inf          # "nekonečno": každá skutečná trasa bude kratší
    best_permutation = []

    start_time = perf_counter()     # spustit stopky

    # itertools.permutations(points) vyrábí jedno po druhém všechna možná pořadí zbylých měst
    for perm in itertools.permutations(points):
        distance = distance_matrix[start, perm[0]]              # úsek z domova do prvního města
        for i in range(len(perm) - 1):
            distance += distance_matrix[perm[i], perm[i + 1]]   # úseky mezi sousedními městy
        distance += distance_matrix[perm[-1], start]            # úsek z posledního města domů

        if distance < best_distance:                            # kratší trasa než dosud -> zapamatuj
            best_distance = distance
            best_permutation = [start] + list(perm) + [start]

    compute_time = perf_counter() - start_time                  # zastavit stopky

    return best_permutation, compute_time

In [ ]:
path, compute_time = brute_force_tsp(distance_matrix, 0)
print("Optimální trasa: ", path, "Délka: ", compute_distance(distance_matrix, path)[0], "Čas: ", compute_time, 's')
tsp_graph_vis(G, pos, path)
plt.title(f'Optimum, délka: {compute_distance(distance_matrix, path)[0]}')

### Jak roste čas s počtem měst?
Spustíme hrubou sílu pro 2, 3, ..., 11 měst a každý běh změříme. (11 měst = 10! = 3 628 800 tras, několik sekund.)

In [ ]:
times = []                                   # sem sbíráme naměřené časy
for i in range(2, 12):                       # i = 2, 3, ..., 11 měst
    _, _, distance_matrix_test = tsp_graph(i)              # mapa s i městy
    _, compute_time = brute_force_tsp(distance_matrix_test, 0)
    print(f'Měst: {i}, čas výpočtu: {compute_time} s')
    times.append(compute_time)
plt.plot(range(2, 12), times)                # lineární graf: "nic, nic, nic, stěna"
plt.xlabel("počet měst"); plt.ylabel("čas [s]")

## 3. Kam až hrubá síla dosáhne? Extrapolace do stáří vesmíru

Naměřené časy rostou jako (n−1)! – počet tras, které musíme vyhodnotit.
Z největší změřené instance odhadneme čas na **jednu trasu** a extrapolujeme: kolik měst zvládneme,
než výpočet potrvá déle, než uplynulo od **velkého třesku** (≈ 13,8 miliardy let)?

In [ ]:
brute_force_times = dict(zip(range(2, 12), times))   # měření z buňky výše: {2: t2, 3: t3, ..., 11: t11}

YEAR = 365.25 * 24 * 3600                 # sekund v roce
AGE_OF_UNIVERSE = 13.8e9 * YEAR           # ≈ 4,35e17 sekund od velkého třesku


def human_time(seconds: float) -> str:
    """Převede počet sekund na čitelný text: '3.2 ms', '1.7 days', '6.4 thousand years', ..."""
    units = [("ns", 1e-9), ("µs", 1e-6), ("ms", 1e-3), ("s", 1), ("min", 60), ("h", 3600),
             ("days", 86400), ("years", YEAR), ("thousand years", 1e3 * YEAR),
             ("million years", 1e6 * YEAR), ("billion years", 1e9 * YEAR)]
    if seconds > 1e6 * 1e9 * YEAR:                       # absurdně velké -> vědecký zápis
        return f"{seconds / (1e9 * YEAR):.1e} billion years"
    for name, size in reversed(units):                   # najdi největší jednotku, která se vejde
        if seconds >= size:
            return f"{seconds / size:,.1f} {name}"
    return f"{seconds / 1e-9:.2f} ns"


# Čas na jednu trasu = (čas pro 11 měst) / (počet tras pro 11 měst)
n_measured = max(brute_force_times)
time_per_route = brute_force_times[n_measured] / math.factorial(n_measured - 1)
print(f"Změřeno: {n_measured} měst -> {math.factorial(n_measured - 1):,} tras za {brute_force_times[n_measured]:.2f} s")
print(f"=> vyhodnocení jedné trasy trvá ~{time_per_route * 1e9:.0f} ns\n")

# Teď odhadni čas pro n = 2, 3, 4, ... dokud nepřesáhne stáří vesmíru
rows = []
n = 2
while True:
    routes = math.factorial(n - 1)             # (n-1)! tras
    estimate = routes * time_per_route         # odhad v sekundách
    rows.append({"měst": n,
                 "tras (n-1)!": f"{routes:.3e}",
                 "odhad času": human_time(estimate),
                 "násobků stáří vesmíru": f"{estimate / AGE_OF_UNIVERSE:.2e}",
                 "sekund": estimate})
    if estimate > AGE_OF_UNIVERSE:             # skonči, když překročíme stáří vesmíru
        break
    n += 1

extrapolation = pd.DataFrame(rows).set_index("měst")
print(f"Hrubá síla překročí stáří vesmíru při n = {n} městech.")
extrapolation.drop(columns="sekund")

In [ ]:
# Stejná čísla jako graf. Svislá osa je LOGARITMICKÁ: každý dílek je 1000x víc.
plt.figure(figsize=(8, 5))
plt.semilogy(extrapolation.index, extrapolation["sekund"], "o-", label="hrubá síla (n-1)! – extrapolace")
plt.semilogy(list(brute_force_times), list(brute_force_times.values()), "s", color="red", label="změřeno")

# Vodorovné referenční čáry, aby čísla něco znamenala
references = {"1 sekunda": 1, "1 hodina": 3600, "1 rok": YEAR,
              "lidský život (80 let)": 80 * YEAR, "stáří vesmíru": AGE_OF_UNIVERSE}
for label, value in references.items():
    plt.axhline(value, ls="--", color="gray", lw=0.8)
    plt.text(2, value * 1.5, label, fontsize=8, color="gray")

plt.xlabel("počet měst")
plt.ylabel("čas výpočtu [s]")
plt.title("TSP hrubou silou: čas výpočtu vs. počet měst")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()

### Pomohl by rychlejší počítač?

Předpokládejme počítač 1000×, 1 000 000× nebo 1 000 000 000× rychlejší
(poslední je zhruba *všechny počítače na Zemi dohromady*).
O kolik měst víc zvládneme za hodinu, za rok, za stáří vesmíru?

In [ ]:
def max_cities(time_budget: float, speedup: float = 1.0) -> int:
    """Největší počet měst, jejichž řešení hrubou silou se vejde do `time_budget` sekund na `speedup`x rychlejším stroji."""
    n = 2
    # přidávej města, dokud se odhad času vejde do rozpočtu  ((n+1) měst -> n! tras)
    while math.factorial(n) * time_per_route / speedup <= time_budget:
        n += 1
    return n

budgets = {"1 hodina": 3600, "1 rok": YEAR, "stáří vesmíru": AGE_OF_UNIVERSE}
speedups = [1, 1e3, 1e6, 1e9]

table = pd.DataFrame({f"{s:,.0f}x rychlejší": [max_cities(b, s) for b in budgets.values()] for s in speedups},
                     index=budgets.keys())
table.index.name = "časový rozpočet"
print("Maximální počet měst řešitelných hrubou silou:")
table

## 4. Přesně, ale chytřeji: dynamické programování (Held–Karp)

Hrubá síla opakuje spoustu práce: trasy A-B-C-D-E a A-B-C-E-D mají společný začátek A-B-C, a přesto ho sčítá dvakrát.
Held–Karp si pro každou *množinu už navštívených měst* a každé *poslední město* pamatuje nejkratší způsob, jak se tam dostat –
a nikdy nepočítá totéž dvakrát. Složitost je O(n²·2ⁿ) – **stále exponenciální**, ale 2ⁿ roste mnohem pomaleji než n!.
Je to nejlepší známý přesný algoritmus pro obecný TSP.

Poznámka: polynomiální algoritmus pro TSP nikdo nezná – kdybyste ho našli, dokázali byste P = NP
(a vydělali 1 000 000 dolarů – viz notebook `P_vs_NP`).

In [ ]:
def held_karp_tsp(distance_matrix: np.ndarray, start: int) -> tuple[list[int], float]:
    """Přesný TSP dynamickým programováním přes podmnožiny měst."""
    n = distance_matrix.shape[0]
    others = [c for c in range(n) if c != start]   # všechna města kromě startu
    m = len(others)

    start_time = perf_counter()

    # dp[(mask, j)] = délka nejkratší cesty ze `start`, která navštíví přesně města v `mask` a skončí v others[j].
    # `mask` je množina měst zapsaná jako dvojkové číslo: bit j = 1 znamená "others[j] bylo navštíveno".
    dp, parent = {}, {}
    for j in range(m):
        dp[(1 << j, j)] = distance_matrix[start, others[j]]     # jedno navštívené město: jen silnice ze startu

    for mask in range(1, 1 << m):                # projdi všech 2^m podmnožin (všechna dvojková čísla o m bitech)
        for j in range(m):                       # j = město, kde cesta právě končí
            if not (mask >> j) & 1 or (mask, j) not in dp:      # j musí být v množině a záznam musí existovat
                continue
            base = dp[(mask, j)]
            for k in range(m):                   # zkus prodloužit cestu do dosud nenavštíveného města k
                if (mask >> k) & 1:              # k už navštíveno -> přeskoč
                    continue
                new_mask = mask | (1 << k)       # množina s přidaným k
                candidate = base + distance_matrix[others[j], others[k]]
                if candidate < dp.get((new_mask, k), np.inf):   # lepší než dosud -> ulož
                    dp[(new_mask, k)] = candidate
                    parent[(new_mask, k)] = j    # zapamatuj, odkud jsme přišli (pro rekonstrukci trasy)

    full = (1 << m) - 1                          # množina všech měst (všechny bity = 1)
    # uzavři okruh: přičti silnici zpět na start a vyber nejlepší poslední město
    best_j = min(range(m), key=lambda j: dp[(full, j)] + distance_matrix[others[j], start])

    # Rekonstrukce trasy: jdi pozpátku po ukazatelích `parent`
    path, mask, j = [], full, best_j
    while True:
        path.append(others[j])
        previous = parent.get((mask, j))
        if previous is None:
            break
        mask ^= 1 << j                           # odeber j z množiny
        j = previous
    path = [start] + path[::-1] + [start]        # otoč a přidej start na oba konce

    compute_time = perf_counter() - start_time
    return path, compute_time


path, compute_time = held_karp_tsp(distance_matrix, 0)
print("Optimální trasa: ", path, "Délka: ", compute_distance(distance_matrix, path)[0], "Čas: ", compute_time, 's')
tsp_graph_vis(G, pos, path)
plt.title(f'Held-Karp (optimum), délka: {compute_distance(distance_matrix, path)[0]}')

In [ ]:
# Složitost Held–Karpa: změřeno do 15 měst, srovnání s hrubou silou
held_karp_times = {}
for i in range(3, 16):
    _, _, distance_matrix_test = tsp_graph(i)
    _, compute_time = held_karp_tsp(distance_matrix_test, 0)
    held_karp_times[i] = compute_time
    print(f'Měst: {i}, čas výpočtu: {compute_time:.4f} s')

# čas na jednu elementární operaci DP, odhadnutý z největší instance (n^2 * 2^n operací)
n_hk = max(held_karp_times)
time_per_op = held_karp_times[n_hk] / (n_hk ** 2 * 2 ** n_hk)

def brute_force_estimate(n): return math.factorial(int(n) - 1) * time_per_route
def held_karp_estimate(n):   return float(n) ** 2 * 2.0 ** n * time_per_op

n_hk_universe = next(n for n in range(3, 200) if held_karp_estimate(n) > AGE_OF_UNIVERSE)
print(f"\nHeld-Karp překročí stáří vesmíru při n = {n_hk_universe} městech (hrubá síla: n = {len(extrapolation) + 1}).")

sizes = [10, 15, 20, 25, 30, 40, 50, 60, 70]
comparison = pd.DataFrame({
    "hrubá síla": [human_time(brute_force_estimate(n)) for n in sizes],
    "Held-Karp":  [human_time(held_karp_estimate(n))   for n in sizes],
}, index=pd.Index(sizes, name="měst"))

ns = np.arange(2, 80)
plt.figure(figsize=(8, 5))
plt.semilogy(ns, [brute_force_estimate(n) for n in ns], label="hrubá síla  (n-1)!")
plt.semilogy(ns, [held_karp_estimate(n) for n in ns], label="Held-Karp  n^2 * 2^n")
plt.semilogy(list(brute_force_times), list(brute_force_times.values()), "s", color="C0")
plt.semilogy(list(held_karp_times), list(held_karp_times.values()), "s", color="C1")
plt.axhline(AGE_OF_UNIVERSE, ls="--", color="gray"); plt.text(3, AGE_OF_UNIVERSE * 2, "stáří vesmíru", color="gray")
plt.axhline(3600, ls="--", color="gray"); plt.text(3, 3600 * 2, "1 hodina", color="gray")
plt.ylim(1e-6, 1e30)
plt.xlabel("počet měst"); plt.ylabel("čas výpočtu [s]"); plt.grid(True, which="both", alpha=0.3); plt.legend()
plt.title("Přesné algoritmy pro TSP (čtverce = změřeno, čáry = extrapolace)")
plt.show()
comparison

**Závěr:** přesné algoritmy přestávají fungovat někde mezi 20 a 60 městy – rozvozová dodávka má víc zastávek každý den.
Proto se v praxi používají **heuristiky** – rychlé algoritmy, které dají *dobrou* (ne nutně optimální) trasu.
Níže jsou tři klasické.

## 5. Heuristika 1: nejbližší soused (greedy)

„Z aktuálního města jeď vždy do nejbližšího, kde jsi ještě nebyl.“ Extrémně rychlé (zhruba n² operací),
bez záruky – poslední úseky bývají dlouhé, protože zbylá města jsou daleko.

In [ ]:
def greedy_tsp(distance_matrix: np.ndarray, start: int) -> tuple[list[int], float]:
    """Heuristika nejbližšího souseda."""
    points = list(range(distance_matrix.shape[0]))   # města, která ještě zbývá navštívit
    points.remove(start)

    visited = [start]               # trasa, kterou stavíme
    next_point = start              # kde právě stojíme

    start_time = perf_counter()

    while points:                                          # dokud zbývají nenavštívená města
        distances = distance_matrix[next_point, points]    # vzdálenosti z aktuálního města do všech nenavštívených
        min_distance_index = np.argmin(distances)          # pozice té nejmenší
        next_point = points.pop(min_distance_index)        # jeď tam (a odeber město ze seznamu nenavštívených)
        visited.append(next_point)

    visited.append(start)           # návrat domů

    compute_time = perf_counter() - start_time

    return visited, compute_time

In [ ]:
path, compute_time = greedy_tsp(distance_matrix, 0)
print("Trasa: ", path, "Délka: ", compute_distance(distance_matrix, path)[0], "Čas: ", compute_time, 's')
tsp_graph_vis(G, pos, path)
plt.title(f'Nejbližší soused, délka: {compute_distance(distance_matrix, path)[0]}')

In [ ]:
# Složitost nejbližšího souseda: 2..29 měst – vše ve zlomku milisekundy
times = []
for i in range(2, 30):
    _, _, distance_matrix_test = tsp_graph(i)
    _, compute_time = greedy_tsp(distance_matrix_test, 0)
    print(f'Měst: {i}, čas výpočtu: {compute_time} s')
    times.append(compute_time)
plt.plot(range(2, 30), times)
plt.xlabel("počet měst"); plt.ylabel("čas [s]")

## 6. Heuristika 2: minimální kostra (MST) – záruka nejvýš 2× optimum

1. Sestav **minimální kostru**: nejlevnější sadu silnic, která propojí všechna města (bez okruhů).
2. Obejdi strom (do hloubky) a zapisuj města v pořadí první návštěvy.
3. Toto pořadí je trasa. Přeskočení už navštívených měst ji nikdy neprodlouží (trojúhelníková nerovnost).

Proč nejvýš 2× optimum: optimální trasa bez jedné silnice je kostra, tedy MST ≤ OPT; obejití stromu použije každou
silnici dvakrát, tedy ≤ 2·MST ≤ 2·OPT; zkratky to jen zkrátí.

In [ ]:
def mst_approx_tsp(distance_matrix: np.ndarray, start: int) -> tuple[list[int], float]:
    """2-aproximace pomocí minimální kostry."""
    import numpy as np
    from scipy.sparse.csgraph import minimum_spanning_tree   # hotový algoritmus na MST (uvnitř Kruskal/Prim)
    from collections import defaultdict

    num_cities = distance_matrix.shape[0]

    start_time = perf_counter()

    # Krok 1: spočítej minimální kostru (výsledek: matice jen se silnicemi stromu)
    mst_sparse = minimum_spanning_tree(distance_matrix)
    mst_matrix = mst_sparse.toarray()
    mst_matrix += mst_matrix.T          # udělej ji symetrickou (silnice vedou oběma směry)

    # Převeď matici na seznam "kdo je s kým spojen"
    adjacency_list = defaultdict(list)
    for i in range(num_cities):
        for j in range(num_cities):
            if mst_matrix[i, j] > 0:
                adjacency_list[i].append(j)

    # Krok 2: průchod stromem do hloubky, zapisujeme první návštěvu každého města
    visited = []
    visited_set = set()

    def dfs(u):
        visited.append(u)               # první návštěva u -> jde do trasy
        visited_set.add(u)
        for v in adjacency_list[u]:     # pak navštiv všechny sousedy ve stromě ...
            if v not in visited_set:    # ... které jsme ještě neviděli
                dfs(v)

    dfs(start)
    visited.append(start)               # návrat domů  (krok 3 – zkratky – se stane sám od sebe:
                                        #  zapisovali jsme jen první návštěvy, takže "cesta zpět" se přeskočí)

    compute_time = perf_counter() - start_time

    return visited, compute_time

In [ ]:
path, compute_time = mst_approx_tsp(distance_matrix, 0)
print("Trasa: ", path, "Délka: ", compute_distance(distance_matrix, path)[0], "Čas: ", compute_time, 's')
tsp_graph_vis(G, pos, path)
plt.title(f'MST, délka: {compute_distance(distance_matrix, path)[0]}')

In [ ]:
# Složitost MST heuristiky: 2..29 měst
times = []
for i in range(2, 30):
    _, _, distance_matrix_test = tsp_graph(i)
    _, compute_time = mst_approx_tsp(distance_matrix_test, 0)
    print(f'Měst: {i}, čas výpočtu: {compute_time} s')
    times.append(compute_time)
plt.plot(range(2, 30), times)
plt.xlabel("počet měst"); plt.ylabel("čas [s]")

## 7. Heuristika 3: Christofides – záruka nejvýš 1,5× optimum

MST + nejlevnější spárování měst s lichým stupněm ve stromě + Eulerův tah (každá silnice jednou) + zkratky.
Použijeme hotovou implementaci z networkx. 44 let (1976–2020) to byla nejlepší známá záruka.

In [ ]:
start_time = perf_counter()
path = nx_app.christofides(G, weight="weight")      # networkx udělá všechny kroky za nás
compute_time = perf_counter() - start_time
print("Trasa: ", path, "Délka: ", compute_distance(distance_matrix, path)[0], "Čas: ", compute_time, 's')
tsp_graph_vis(G, pos, path)
plt.title(f'Christofides, délka: {compute_distance(distance_matrix, path)[0]}')

## 8. Všechny tři heuristiky na 20 městech
Hrubá síla by tu potřebovala 19!/2 ≈ 6·10¹⁶ tras – nemožné. Heuristiky odpoví za milisekundy.
Porovnejte tři obrázky: hledejte křížící se silnice (známka, že trasu lze ještě zlepšit pomocí „2-opt“).

In [ ]:
cities = 20
G, pos, distance_matrix = tsp_graph(cities)
tsp_graph_vis(G, pos)
plt.title('Problém obchodního cestujícího')

path, compute_time = greedy_tsp(distance_matrix, 0)
plt.figure()
tsp_graph_vis(G, pos, path)
plt.title(f'Nejbližší soused, délka: {compute_distance(distance_matrix, path)[0]:.3f}')

path, compute_time = mst_approx_tsp(distance_matrix, 0)
plt.figure()
tsp_graph_vis(G, pos, path)
plt.title(f'MST, délka: {compute_distance(distance_matrix, path)[0]:.3f}')

path = nx_app.christofides(G, weight="weight")
plt.figure()
tsp_graph_vis(G, pos, path)
plt.title(f'Christofides, délka: {compute_distance(distance_matrix, path)[0]:.3f}')

## Cvičení

Naprogramujte **2-opt**: opakovaně hledejte v trase dvě silnice (a,b) a (c,d) takové, že jejich nahrazení silnicemi (a,c) a (b,d)
trasu zkrátí (to znamená obrátit úsek trasy mezi b a c). Skončete, když žádná taková dvojice neexistuje.
Začněte z trasy nejbližšího souseda a porovnejte výsledek s Christofidesem na 20 a 100 městech.

In [ ]:
def two_opt(distance_matrix: np.ndarray, path: list[int]) -> list[int]:
    """Zlepší uzavřenou trasu pomocí 2-opt výměn. `path` začíná i končí stejným městem."""
    route = path[:-1]               # pracujeme bez opakovaného startu na konci
    improved = True
    while improved:
        improved = False
        # YOUR CODE HERE:
        #   pro každou dvojici pozic i < k:
        #       a, b = route[i], route[i + 1];  c, d = route[k], route[(k + 1) % len(route)]
        #       pokud d(a,c) + d(b,d) < d(a,b) + d(c,d):  obrať route[i+1 : k+1]  a nastav improved = True
        pass
    return route + [route[0]]

# path, _ = greedy_tsp(distance_matrix, 0)
# better = two_opt(distance_matrix, path)
# print("nejbližší soused:", compute_distance(distance_matrix, path)[0], " po 2-opt:", compute_distance(distance_matrix, better)[0])